# FireSystem: How Each Data Source Affects the Hotspot Model

Measures what each data source adds to the hotspot forecast of `XGBoost_Hotspot_Forecast.ipynb`. Every model variant uses **the same rows** (pixel-month panel: every hotspot pixel-month plus 11 sampled pixel-months without one), **the same XGBoost settings** and **the same walk-forward evaluation**: each season is predicted by a model fitted only on earlier seasons. Only the input features change.

| Group | Features | Known before the month by |
|---|---|---|
| Leaf area | MCD15A3H monthly leaf area 24–35 months earlier, and its mean / min / max / spread / valid months | 24 months |
| Terrain | SRTM elevation and slope of the 500 m pixel | fixed |
| Land cover | MCD12Q1 IGBP class of the latest year ≤ row year − 2 | 13 months |
| NDVI / EVI | MOD13A1 monthly NDVI and EVI 24–35 months earlier, with the same statistics | 24 months |
| Burned area | MCD64A1 + VNP64A1: pixel burned 12 and 24 months earlier, months burned and 3 × 3 neighbourhood burns over 12–23 months earlier | 12 months |

Variants: leaf area alone, the current model (leaf area + terrain), the current model plus each new source on its own, and everything together. Two baselines (hotspot in the same month last year; the month's hotspot share) give the floor. Validation seasons and test seasons are reported separately; a source helps if it improves both.

## 1. Set Up

In [ ]:
import json
import sys
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils import chile_regions as cr
from src.utils import fire_datasets as fd
from src.utils import forecasting as fc
from src.utils import geotiff_layers as gl
from src.utils import hotspots as hs

try:
    import xgboost as xgb
except Exception as error:  # macOS needs the OpenMP runtime
    raise ImportError("XGBoost could not be loaded. On macOS install the OpenMP runtime: brew install libomp") from error
print("xgboost", xgb.__version__)

## 2. Set Parameters
Uses the data already downloaded by `XGBoost_Hotspot_Forecast.ipynb` (hotspots, leaf area, elevation) and `AppEEARS_API_Area.ipynb` (land cover, NDVI / EVI, burned area). Nothing is downloaded here.

In [ ]:
REGION = "Maule"

In [ ]:
SLUG = cr.region_slug(REGION)
PARENT_REGION = cr.REDUCED_REGIONS.get(REGION, (REGION, 1.0))[0]
DATA_DIR = REPO_ROOT / "data"
APPEEARS_DIR = DATA_DIR / "AppEEARS" / cr.region_slug(PARENT_REGION)
LAI_DIR = APPEEARS_DIR / "extra"
FIRMS_PATHS = sorted((DATA_DIR / "FIRMS" / "archive").glob("*_Chile.csv")) + sorted((DATA_DIR / "FIRMS" / "api" / SLUG).glob("api_*.csv"))
OUTPUT_DIR = DATA_DIR / "models" / "hotspots" / SLUG

START = date(2012, 3, 1)
LAST_MONTH = (date.today().replace(day=1) - timedelta(days=1)).replace(day=1)
LAI_LAGS = range(24, 36)
VEG_LAGS = range(24, 36)              # NDVI / EVI, same window as leaf area
BURN_LAGS = (12, 24)
BURN_WINDOW = range(12, 24)
LAND_COVER_LAG_YEARS = 2
NEGATIVES_PER_POSITIVE = 11
SEED = 42
SEASON_START_MONTH = 7
VAL_SEASONS = 2
TEST_SEASONS = 2

# Same settings for every variant: the parameters chosen by the forecast notebook's tuning, if saved
PARAMS = {"max_depth": 6, "learning_rate": 0.1, "n_estimators": 300, "min_child_weight": 10.0,
          "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 1.0}
reports = sorted(OUTPUT_DIR.glob(f"report_hotspots_{SLUG}_*.json"))
if reports:
    PARAMS = json.loads(reports[-1].read_text())["best_params"]
print("XGBoost parameters:", PARAMS)


def make_model() -> xgb.XGBClassifier:
    return xgb.XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist", enable_categorical=True,
        random_state=SEED, n_jobs=-1, **PARAMS,
    )

## 3. Build the Panel and Grids

In [ ]:
chile = cr.load_regions(DATA_DIR / "boundaries" / cr.BOUNDARIES_FILE)
region = cr.named_region_geometry(chile, REGION)
months = pd.date_range(START, LAST_MONTH, freq="MS")

lai = gl.open_lai(LAI_DIR)
lat, lon = lai["lat"].values, lai["lon"].values
monthly_lai = gl.monthly_mean(lai, "lai", months)
valid = fd.region_mask(lai, region) & np.isfinite(monthly_lai.values).any(axis=0)

spots = hs.load_hotspots(FIRMS_PATHS, region=region)
grid = hs.monthly_hotspot_grid(spots, lat, lon, months, api_exclude=hs.static_source_mask(FIRMS_PATHS, lat, lon))
first_month = pd.Timestamp(START) + pd.DateOffset(months=max(LAI_LAGS))
panel, NEGATIVE_RATE = hs.sample_panel(grid, valid, first_month, NEGATIVES_PER_POSITIVE, SEED)
print(f"Panel: {len(panel):,} rows ({panel['hotspot'].sum():,} with a hotspot), {first_month:%Y-%m} .. {months[-1]:%Y-%m}")


def same_grid(ds) -> bool:
    return ds.sizes["lat"] == len(lat) and ds.sizes["lon"] == len(lon) and np.allclose(ds["lat"], lat) and np.allclose(ds["lon"], lon)


# Terrain
terrain = gl.terrain_on_grid(gl.open_elevation(LAI_DIR), lat, lon)

# Land cover (MCD12Q1 LC_Type1)
land_cover = fd.land_cover_grid(fd.open_product("MCD12Q1.061", APPEEARS_DIR))["land_cover_igbp"].load()

# NDVI / EVI (MOD13A1, good-reliability pixels), monthly means
veg_spec = fd.DATASETS["MODIS_vegetation"]
veg_ds = fd.open_product(veg_spec["product"], APPEEARS_DIR)
veg = fd.vegetation_grid(veg_ds, veg_spec["good_reliability"])
monthly_ndvi = gl.monthly_mean(veg, "ndvi", months)
monthly_evi = gl.monthly_mean(veg, "evi", months)

# Burned area: a pixel-month is burned if MODIS or VIIRS says so
burn_histories = [fc.monthly_burn_history(fd.open_product(fd.DATASETS[f"{s}_burned_area"]["product"], APPEEARS_DIR)) for s in ("MODIS", "VIIRS")]
burn_history = burn_histories[0].copy(data=np.fmax(burn_histories[0].values, burn_histories[1].values))

for name, ds in {"land cover": land_cover, "NDVI/EVI": veg_ds, "burned area": burn_histories[0]}.items():
    if not same_grid(ds):
        raise ValueError(f"{name} is not on the leaf area grid")
print(f"Land cover {pd.DatetimeIndex(land_cover['time'].values).year.min()}..{pd.DatetimeIndex(land_cover['time'].values).year.max()} | "
      f"NDVI/EVI {pd.Timestamp(veg_ds['time'].values.min()):%Y-%m}..{pd.Timestamp(veg_ds['time'].values.max()):%Y-%m} | "
      f"burned area {burn_history.indexes['month'].min():%Y-%m}..{burn_history.indexes['month'].max():%Y-%m}")

## 4. Build Every Feature Group

In [ ]:
i, j = panel["lat_idx"].to_numpy(dtype=int), panel["lon_idx"].to_numpy(dtype=int)
burn = fc.add_burn_lags(panel, burn_history, lags=BURN_LAGS, window=BURN_WINDOW)
blocks = {
    "leaf_area": hs.lai_features(panel, monthly_lai, LAI_LAGS),
    "terrain": pd.DataFrame({"elevation_m": terrain["elevation_m"].values[i, j], "slope_deg": terrain["slope_deg"].values[i, j]}, index=panel.index),
    "land_cover": hs.land_cover_features(panel, land_cover, LAND_COVER_LAG_YEARS)[["land_cover"]],
    "ndvi_evi": pd.concat([hs.grid_lag_features(panel, monthly_ndvi, VEG_LAGS, "ndvi"),
                           hs.grid_lag_features(panel, monthly_evi, VEG_LAGS, "evi")], axis=1),
    "burned_area": burn,
}
GROUPS = {name: list(block.columns) for name, block in blocks.items()}
features = pd.concat([panel, *blocks.values(), hs.hotspot_lag_features(panel, grid, [12])], axis=1)
features["cal_month"] = pd.DatetimeIndex(features["date"]).month
df, split_seasons = fc.chronological_split(features, VAL_SEASONS, TEST_SEASONS, SEASON_START_MONTH)

pd.DataFrame({
    "features": {g: len(c) for g, c in GROUPS.items()},
    "missing_share": {g: float(df[c].isna().to_numpy().mean()) for g, c in GROUPS.items()},
}).round(3)

In [ ]:
# Hotspot share per land cover class (sampled rows; 1 hotspot : 11 without)
by_class = df.groupby("land_cover", observed=True)["hotspot"].agg(rows="size", hotspot_share="mean")
by_class.insert(0, "class", by_class.index.map(hs.IGBP_CLASSES))
by_class.sort_values("hotspot_share", ascending=False).round(4)

## 5. Walk-Forward Evaluation of Each Variant
Each variant is predicted season by season: validation seasons with models fitted on the training seasons before them, test seasons with models fitted on every earlier season.

In [ ]:
CURRENT = ["leaf_area", "terrain"]
VARIANTS = {
    "leaf area": ["leaf_area"],
    "current (leaf area + terrain)": CURRENT,
    "current + land cover": [*CURRENT, "land_cover"],
    "current + NDVI/EVI": [*CURRENT, "ndvi_evi"],
    "current + burned area": [*CURRENT, "burned_area"],
    "all": list(GROUPS),
}
eval_seasons = {"validation": split_seasons["val"], "test": split_seasons["test"]}

rows, per_season = [], []
for name, groups in VARIANTS.items():
    cols = [c for g in groups for c in GROUPS[g]]
    for split, seasons in eval_seasons.items():
        predictions = fc.walk_forward(df, cols, "hotspot", make_model, seasons)
        scores = fc.score_predictions(predictions)
        rows.append({"variant": name, "split": split, "features": len(cols), **scores.loc["all", ["roc_auc", "average_precision", "log_loss", "brier"]]})
        per_season += [{"variant": name, "season": s, "roc_auc": scores.loc[s, "roc_auc"]} for s in seasons]
    print(f"done: {name}")

for name, (factory, cols) in {
    "baseline: same month last year": (lambda: fc.SeasonalNaive("hotspot_lag_12"), ["hotspot_lag_12"]),
    "baseline: month's hotspot share": (lambda: fc.Climatology(by=("cal_month",)), ["cal_month"]),
}.items():
    for split, seasons in eval_seasons.items():
        scores = fc.score_predictions(fc.walk_forward(df, cols, "hotspot", factory, seasons))
        rows.append({"variant": name, "split": split, "features": 1, **scores.loc["all", ["roc_auc", "average_precision", "log_loss", "brier"]]})

results = pd.DataFrame(rows)

In [ ]:
# Scores per variant; delta columns compare with the current model (positive = better for AUC, negative = better for log loss)
table = results.pivot(index="variant", columns="split", values=["roc_auc", "average_precision", "log_loss"]).astype(float)
table.columns = [f"{metric}_{split}" for metric, split in table.columns]
current = table.loc["current (leaf area + terrain)"]
for split in eval_seasons:
    table[f"delta_roc_auc_{split}"] = table[f"roc_auc_{split}"] - current[f"roc_auc_{split}"]
    table[f"delta_pr_auc_{split}"] = table[f"average_precision_{split}"] - current[f"average_precision_{split}"]
order = [*VARIANTS, "baseline: month's hotspot share", "baseline: same month last year"]
table = table.loc[order, [
    "roc_auc_validation", "delta_roc_auc_validation", "roc_auc_test", "delta_roc_auc_test",
    "average_precision_validation", "delta_pr_auc_validation", "average_precision_test", "delta_pr_auc_test",
    "log_loss_validation", "log_loss_test",
]]
table.round(4)

In [ ]:
# ROC AUC per season
pd.DataFrame(per_season).pivot(index="variant", columns="season", values="roc_auc").loc[list(VARIANTS)].astype(float).round(4)

## 6. Feature Importance with Every Source
Gain share per data source and the top features, for a model with every group fitted on the training and validation seasons (test seasons left out).

In [ ]:
all_cols = [c for g in GROUPS.values() for c in g]
fit = df[df["split"] != "test"]
model = make_model().fit(fit[all_cols], fit["hotspot"])
gain = pd.Series(model.get_booster().get_score(importance_type="gain")).reindex(all_cols).fillna(0)
gain = gain / gain.sum()
group_of = {c: g for g, cols in GROUPS.items() for c in cols}
print(gain.groupby(gain.index.map(group_of)).sum().sort_values(ascending=False).round(3).rename("gain_share").to_string())
gain.sort_values(ascending=False).head(15).round(4).rename("gain_share").to_frame()

## 7. Save the Comparison

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
path = OUTPUT_DIR / f"feature_impact_{SLUG}_{months[0]:%Y%m}_{months[-1]:%Y%m}.csv"
table.to_csv(path)
print("Saved", path)